Teste função um por um 

In [1]:
from pymongo import MongoClient
import hashlib
import json

In [2]:
from qdrant_client.models import Prefetch, FusionQuery, Fusion, models
from qdrant_client import QdrantClient
from qdrant_client.models import Distance, VectorParams, SparseVectorParams
from qdrant_client.models import PointStruct, SparseVector

In [ ]:
def conectar_banco() -> list[str]:
    client = MongoClient("mongodb://localhost:27017/")
    db = client["TCC"]
    professores = db["Docentes"]
    perfis = list(professores.find({}))
    return perfis 

In [44]:
professores = conectar_banco()

In [5]:
def extrair_areas(professor):
    competencias = professor.get("competencias", {})
    areas = competencias.get("areas", [])

    grandes_areas = set()
    areas_atuacao = set()
    subareas = set()

    for item in areas:
        grande_area = item.get("grande_area")
        area = item.get("area")
        subarea = item.get("subarea")

        if grande_area:
            grandes_areas.add(grande_area)

        if area:
            areas_atuacao.add(area)

        if subarea:
            subareas.add(subarea)

    return {
        "grandes_areas": sorted(grandes_areas),
        "areas": sorted(areas_atuacao),
        "subareas": sorted(subareas)
    }

In [47]:
def montar_texto_projeto(projeto):
    titulo = projeto.get("titulo", "")
    descricao = projeto.get("descricao", "")
    area = projeto.get("areaConhecimento", "")
    
    partes = []

    if titulo:
        partes.append(f"Título: {titulo}")

    if descricao:
        partes.append(f"Descrição: {descricao}")

    if area:
        partes.append(f"Área de conhecimento: {area}")

    return "\n".join(partes)

In [55]:
def montar_componentes_perfil(prof):

    competencias = prof.get("competencias", {})
    iniciacoes = []

    for ic in prof.get("iniciacaoCientifica", []):
        titulo = ic.get("titulo", "")
        resumo = ic.get("resumo", "")
        palavras = ic.get("palavrasChaves", [])

        texto_ic = f"""
                Título: {titulo}

                Resumo: {resumo}

                Palavras-chave: {", ".join(palavras)}
                """.strip()

        iniciacoes.append(texto_ic)

    subareas = [
        a.get("subarea")
        for a in competencias.get("areas", [])
        if a.get("subarea")
    ]
    especialidades = [
        a.get("especialidade")
        for a in competencias.get("areas", [])
        if a.get("especialidade")
    ]
    
    linhas = competencias.get("linhas_pesquisa", [])

    palavras_chave = competencias.get("palavras_chave", [])

    disciplinas = [
        d.get("nome")
        for d in prof.get("disciplinasSigaa", [])
        if d.get("nome")
    ]
    ## bolar o que colocar em cada um desses muita coisa daria para entrar tipo 
    projetos = [
        p.get("titulo")
        for p in prof.get("projetos", [])
        if p.get("titulo")
    ]

    producoes = [
        p.get("titulo")
        for p in prof.get("producoes", {}).get("artigo_periodico", [])
        if p.get("titulo")
    ]

    orientacoes = [
        o.get("titulo")
        for o in prof.get("orientacoes", [])
        if o.get("titulo")
    ]

    return {

        "areas_atuação" : "; ".join(subareas),

        "especialidade" : "; ".join(especialidades),

        "pesquisa": "; ".join(
            linhas +
            palavras_chave
        ),

        "projetos": "; ".join(projetos),

        "producoes": "; ".join(producoes),

        "disciplinas": "; ".join(disciplinas),

        "orientacoes": "; ".join(orientacoes),
        
        "iniciacao_cientifica": "\n\n".join(iniciacoes),
    }

In [ ]:
def montagem_perfil(prof) -> str:
    nomes_disciplinas = [d["nome"] for d in prof.get("disciplinasSigaa", [])]
    nomes_pesquisa = [d["subarea"] for d in prof.get("areas", [])]
    
    disciplinas_text = ", ".join(nomes_disciplinas)
    
    return (
        f""" Nome: {prof.get('nome', '')} Disciplinas: {disciplinas_text.lower()} Projetos: {}"""
        )

In [19]:
def transformar_bytes(texto : str) -> bytes:
    t = texto
    return t.encode("utf-8")

In [ ]:
def criar_perfils_texto_completo(professores) -> list[str]:
    professores_texto = []
    for professor in professores:
        perfil_embedding = montagem_perfil(professor)
        professor = {
            "id" : professor.get("siape"),
            "nome" : professor.get("nome"),
            "texto_embedding" : perfil_embedding,
            "hash" : hashlib.sha256(transformar_bytes(perfil_embedding)).hexdigest(),
            "metadata" : {
                "unidade" : professor.get("unidade"),
               
            }
        }
        professores_texto.append(professor)
        
    return professores_texto


In [ ]:
perfis = criar_perfils_componentes(professores)

In [ ]:
print(perfis)

In [ ]:
from rank_bm25 import BM25Okapi

In [ ]:
corpus = [perfil.get("texto_embedding") for perfil in perfis]
tokenized_corpus = [corpu.split(" ") for corpu in corpus]

bm25 = BM25Okapi(tokenized_corpus)

In [ ]:
query = "compatibilidade eletromagnética"
tokenized_query = query.split(" ")
print(tokenized_query)

In [ ]:
doc_scores = bm25.get_scores(tokenized_query)
print(doc_scores)

In [ ]:
bm25.get_top_n(tokenized_query, corpus, n=1)

In [7]:
from FlagEmbedding import BGEM3FlagModel

In [20]:
# bge-m3 vai ficar na mémoria da nossa aplicação
def carregar_bgeM3():
    return BGEM3FlagModel('BAAI/bge-m3', use_fp16=True)

def rodar_bgeM3(modelo, componentes):
    embeddings = {}
    for nome, texto in componentes.items():
        if not texto or not texto.strip():
            continue

        embeddings[nome] = modelo.encode(
            texto,
            return_dense=True, 
            return_sparse=True
        )

    return embeddings

def obter_denso(perfis_transformados):
    return perfis_transformados['dense_vecs']
    

In [10]:
modelo = carregar_bgeM3()

Reconstructing (incomplete total...): |          |  0.00B /  0.00B            

Fetching 30 files:   0%|          | 0/30 [00:00<?, ?it/s]

Loading weights:   0%|          | 0/391 [00:00<?, ?it/s]

In [11]:
def gerar_hash(componentes):   
    json_normalizado = json.dumps(
        componentes,
        sort_keys=True,
        ensure_ascii=False,
        separators=(",", ":")
    )
    return hashlib.sha256(transformar_bytes(json_normalizado)).hexdigest()

In [58]:
def criar_perfils_componentes(professores) -> list[str]:
    professores_texto = []
    for professor in professores:
        componentes = montar_componentes_perfil(professor)
        professor = {
            "id" : professor.get("siape"),
            "nome" : professor.get("nome"),
            "componentes" : componentes,
            # pensar hash com componentes separados
            "hash" : {
                nome : gerar_hash(componente)
                for nome, componente in componentes.items()
                
            },
            "metadata" : {
                **extrair_areas(professor),
                "unidade" : professor.get("unidade"),
               
            }
        }
        professores_texto.append(professor)
        
    return professores_texto


In [59]:
textos_professores = {}
professores_transformados = criar_perfils_componentes(professores)

In [61]:
for prof in professores_transformados:
    componentes = prof.get("componentes", {})
    hash = prof.get("hash", {})
    #texto_transformado = rodar_bgeM3(modelo,componentes)
    
    #textos_professores[prof["siape"]] = texto_transformado

In [76]:
print(textos_professores)

{'2019312': {'areas atuação': {'dense_vecs': array([-0.04158271,  0.00181272,  0.0035669 , ...,  0.01588366,
        0.01562025, -0.01436381], shape=(1024,), dtype=float32), 'lexical_weights': defaultdict(<class 'int'>, {'167190': np.float32(0.17109384), '232': np.float32(0.06517205), '91969': np.float32(0.19899973), '1140': np.float32(0.07831283), '313': np.float32(0.059441727), '61611': np.float32(0.21481442), '74': np.float32(0.09773798), '35007': np.float32(0.08552919), '849': np.float32(0.102444306), '82512': np.float32(0.13619103), '28': np.float32(0.10183886), '27967': np.float32(0.14433903), '12701': np.float32(0.003936518), '4448': np.float32(0.032358337), '191754': np.float32(0.029244453), '7': np.float32(0.05675175), '39018': np.float32(0.03077953), '182142': np.float32(0.031992033), '284': np.float32(0.031131044), '1514': np.float32(0.108838595), '61898': np.float32(0.034507636), '17632': np.float32(0.082205355), '113288': np.float32(0.0039048232), '123996': np.float32(0.05

In [ ]:
dense = obter_denso(texto_transformado)
print(dense)

In [ ]:
def obter_lexical(perfis_transformados):
    return perfis_transformados["lexical_weights"]

In [ ]:
lexical = obter_lexical(texto_transformado)

In [ ]:
def criar_hash(texto: str) -> str:
    return hashlib.sha256(texto.encode("utf-8")).hexdigest()

In [ ]:
import uuid
def criar_id(professor_id,componente):
    str(uuid.uuid5(
            uuid.NAMESPACE_DNS,
            f"{professor_id['id']}_{componente}"
                )
        )

In [ ]:
def conectar_qdrant(url):
    return QdrantClient(url=url)

def criar_banco_qdrant(client_qdrant):
    client_qdrant.create_collection(
        collection_name="professores",
        vectors_config={
            "dense": VectorParams(size=1024, distance=Distance.COSINE)},
        sparse_vectors_config={
            "sparse": SparseVectorParams(),
        },
    )

def adicionar_professores_qdrant(client_qdrant, perfis, embeddings_professores, lexical_professores):
    points = []
    for professor in perfis:
        
        professor_id  = professor["id"]
    
        embeddings = embeddings_professores[professor_id]
    
        for componente, resultado in embeddings.items:
            
            sparse = resuçtado[professor_id][componente]
            
            sparse_indices = [int(k) for k in sparse.keys()]
            sparse_values = [float(v) for v in sparse.values()]

            points.append(
                PointStruct(
                    id= criar_id(professor_id,componente),
                    vector={
                        "dense": resultado["dense"].tolist(),
                        "sparse": SparseVector(indices=sparse_indices, values=sparse_values),
                    },
                    # metadados
                    payload={
                        "nome": professor.get("nome"),
                        "unidade": professor.get("unidade"),
                        
                    }
                )
            )
            
        client_qdrant.upsert(collection_name="professores", points=points)

def verificar_alteracao_hash(perfils,client_qdrant):
    # fazer consulta no Qdrant e verificar alteração no hash ou não
    perfis_para_atualizar = []
    for perfil in perfils:
        busca = client_qdrant.retrieve(
            collection_name="professores",
            ids=[int(perfil.get("id"))],
            with_payload=True
        )
        if busca:
            payload_qdrant = busca[0].payload
            hash_salvo = payload_qdrant.get("hash")
            versao_salva = payload_qdrant.get("versão")
            
            if hash_salvo != perfil.get("hash") or versao_salva != perfil.get("versão"):
                perfis_para_atualizar.append(perfil)
        
        else:
            perfis_para_atualizar.append(perfil)
        
    return perfis_para_atualizar

In [ ]:
client_qdrant = conectar_qdrant("http://localhost:6333")

In [ ]:
client_qdrant.delete_collection("professores")

In [ ]:
criar_banco_qdrant(client_qdrant)

In [ ]:
adicionar_professores_qdrant(client_qdrant, perfis, dense, lexical)

In [ ]:
def montagem_perfil(prof) -> str:
    nomes_disciplinas = [d["nome"] for d in prof.get("disciplinasSigaa", [])]
    disciplinas_text = ", ".join(nomes_disciplinas)
    
    return (
        f""" Nome : {prof.get('nome', '')} Disciplinas : {disciplinas_text.lower()}"""
        )

In [ ]:
def criar_perfils(professores) -> list[str]:
    professores_texto = []
    for professor in professores:
        perfil_embedding = montagem_perfil(professor)
        professor = {
            "id" : professor.get("siape"),
            "nome" : professor.get("nome"),
            "texto_embedding" : perfil_embedding,
            "hash" : hashlib.sha256(transformar_bytes(perfil_embedding)).hexdigest(),
            "metadata" : {
                **extrair_areas(professor),
                "unidade" : professor.get("unidade"),  
            }
        }
        professores_texto.append(professor)
        
    return professores_texto


In [ ]:
perfisv1 = criar_perfils(professores)

In [ ]:
perfis_para_atualizar = []
for perfil in perfisv1:
        busca = client_qdrant.retrieve(
            collection_name="professores",
            ids=[int(perfil.get("id"))],
            with_payload=True
        )
        
        if busca:
            payload_qdrant = busca[0].payload
            hash_salvo = payload_qdrant.get("hash")
            versao_salva = payload_qdrant.get("versão")
                    
            if hash_salvo != perfil.get("hash") or versao_salva != perfil.get("versão"):
                perfis_para_atualizar.append(perfil)
        
nome = []
for perfil in perfis_para_atualizar:
    nome.append(perfil.get("nome"))
print(nome)

In [ ]:
perfis_atualizar = verificar_alteracao_hash(perfisv1, client_qdrant)

In [ ]:
print(perfis_atualizar)

In [ ]:
texto_transformado = rodar_bgeM3(modelo,perfis_atualizar)

In [ ]:
dense = obter_denso(perfis_atualizar)
lexical = obter_lexical(perfis_atualizar)

In [ ]:
adicionar_professores_qdrant(client_qdrant, perfis_atualizar, dense, lexical)

In [ ]:
def carregar_bgeM3():
    return BGEM3FlagModel('BAAI/bge-m3', use_fp16=True)

In [ ]:
def aluno_rodar_bgem3_denso_lexical(query_aluno, model) -> list:
    aluno = model.encode([query_aluno.lower()], return_dense=True, return_sparse=True)
    embedding_aluno = aluno['dense_vecs'][0]
    lexical_aluno = aluno['lexical_weights'][0]
    return [embedding_aluno, lexical_aluno]

In [ ]:
model = carregar_bgeM3()
query_aluno ="Descrição : Gostaria de desenvolver um projeto que comparasse o desempenho de redes 5G e 5.5G, analisando aspectos como velocidade de conexão e latência Ferramentas : Python, Matlab  Interesses : Telecomunicações, Rede de Computadores, IoT"

In [ ]:
aluno_rodar_bgem3_denso_lexical(query_aluno, model)

In [ ]:
def aluno_rodar_bgem3(query_aluno, model):
    return model.encode([query_aluno], return_dense=True, return_sparse=True)

def aluno_obter_lexical(aluno):
    return aluno['lexical_weights'][0]

def aluno_obter_denso(aluno):
    return aluno['dense_vecs'][0]

In [ ]:
aluno = aluno_rodar_bgem3(query_aluno, model)

In [ ]:
aluno_rodar_bgem3(query_aluno, model)

In [ ]:
aluno_obter_lexical(aluno)

In [ ]:
def busca_densa(client_qdrant,embedding_aluno,collection = "professores"):
    return client_qdrant.query_points(
        collection,
        query=embedding_aluno,
        using="dense",
        limit=10,
    )
    
def busca_esparsa(client_qdrant, lexical_aluno):
    sparse_query = SparseVector(
    indices=[int(k) for k in lexical_aluno.keys()],
    values=[float(v) for v in lexical_aluno.values()],
    )

    return client_qdrant.query_points(
        "professores",
        query=sparse_query,
        using="sparse",
        limit=10,
    )

In [ ]:
results = busca_densa(client_qdrant,aluno_obter_denso(aluno),collection = "professores")

ranking_densa = []
for point in results.points:
    print(point.payload["nome"], point.score)
    
    ranking_densa.append([point.payload["nome"], point.score])

In [ ]:
results = busca_esparsa(client_qdrant,aluno_obter_lexical(aluno))


ranking_lexical = []
for point in results.points:
    print(point.payload["nome"], point.score)
    
    ranking_lexical.append([point.payload["nome"], point.score])

In [ ]:
chunks_gustavo = [
    {"tipo": "perfil",
     "texto": "Possui graduação em Engenharia Elétrica... circuitos em ultra-baixa potência "
              "para aplicações biomédicas. ASDM, Halo-implantes, Modulação."},

    # projetos
    {"tipo": "projeto",
     "texto": "Estudo e desenvolvimento de redes neurais otimizadas para detecção de "
              "batimentos ventriculares prematuros (PVC)."},
    {"tipo": "projeto",
     "texto": "Implementação de Circuitos Neuromórficos Digitais utilizando a Arquitetura "
              "Extreme Learning Machine. "},   # same ELM description as Odilon's
    {"tipo": "projeto",
     "texto": "Análise de viabilidade de projeto e implementação de processadores RISC-V "
              "em FPGA. "},
    {"tipo": "projeto",
     "texto": "Desenvolvimento de um Sistema de Monitoramento Cardíaco em FPGA Visando "
              "Aplicações em IoT. "},
    {"tipo": "projeto",
     "texto": "Reconfiguração do processador softcore RVfpga-EL2 para Internet das Coisas."},

    # orientacoes
    {"tipo": "orientacao", "texto": "Uma metodologia de projeto de filtros Butterworth passa-baixa utilizando FDDTAs"},
    {"tipo": "orientacao", "texto": "Uma topologia CMOS Nauta OTA sem Calibração em Ultra-Baixa Tensão e Ultra-Baixa Potência"},
    {"tipo": "orientacao", "texto": "Utilização do Filtro de Savitzky-Golay no Cancelamento de interferência de 60-Hz em Sinais de Eletrocardiograma"},
    {"tipo": "orientacao", "texto": "Uma metodologia de projeto de conversor A/D SAR-PWM híbrido"},

    # tccs — this is where the plant/agriculture and health evidence actually lives
    {"tipo": "tcc", "texto": "Sistema de monitoramento remoto de pacientes e pessoas vulneráveis"},
    {"tipo": "tcc", "texto": "Sensor de Irradiação Solar Usando Termistores"},
    {"tipo": "tcc", "texto": "Implementação de um Toolbox/Simulink para a Modelagem de um Conversor A/D do Tipo SAR-PWM"},
    {"tipo": "tcc", "texto": "Protótipo de Estação Meteorológica Didática"},
    {"tipo": "tcc", "texto": "Pulseira de emergência"},
    {"tipo": "tcc", "texto": "Sistema de Administração de Medicamentos"},
    {"tipo": "tcc", "texto": "Aprimorando e Inovando Aplicações de Microcontroladores em ambiente educacional"},
    {"tipo": "tcc", "texto": "Medidor Multiparâmetro de Sinais Vitais"},
    {"tipo": "tcc", "texto": "Estudo e otimização da Eficiência Energética em um Sistema de Refrigeração Industrial"},
    {"tipo": "tcc", "texto": "Projeto de Módulo Educacional de Aquisição de Dados para Agricultura utilizando Tecnologia LoRa"},
    {"tipo": "tcc", "texto": "Controle e monitoramento de ciclo hidropônico do tipo NFT"},
    {"tipo": "tcc", "texto": "Monitoramento de motor de indução trifásico baseado em sensor IoT"},
    {"tipo": "tcc", "texto": "Sistema de Monitoramento de Consumo de Gás de Cozinha"},
    {"tipo": "tcc", "texto": "Síntese e Implementação de um Núcleo RISC-V RV32IM Utilizando o Processo SkyWater 130 nm"},

    # iniciacaoCientifica — has real text here, not just titles
    {"tipo": "ic", "texto": "Multiplicadores binários em arquitetura RISC-V"},
    {"tipo": "ic", "texto": "Estudo e desenvolvimento de uma microarquitetura RISC-V baseada no ISA RV32I"},
    {"tipo": "ic", "texto": "Estudo e desenvolvimento de periféricos internos para trabalhar em conjunto com "
                             "arquiteturas RISC-V baseadas no ISA RV32I"},
    {"tipo": "ic", "texto": "SÍNTESE DE MICROARQUITETURA RISC-V RV32IM. Arquitetura e Organização de "
                             "Computadores, Computação de Alto Desempenho, RISC-V. "},
    {"tipo": "ic", "texto": "ESTUDO DE VIABILIDADE DE IMPLEMENTAÇÃO DE UM SOFTCORE RISC-V MULTI-CICLO EM FPGA. "
                             "Análise de desempenho, Arquitetura de computadores, RISC-V, Verilog HDL. "},
    {"tipo": "ic", "texto": "MULTIPLICADORES BINÁRIOS EM MICROARQUITETURA RISC-V. Arquitetura e Organização de "
                             "Computadores, Computação de Alto Desempenho, RISC-V. "},
]